## Teen Phone Addiction Dataset- Exploratory Data Analysis
### OBJECTIVE- We try to analyse the dataset given: phone addiction in teens, and related factors.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display 

: 

In [ ]:
df_raw = pd.read_csv("teen_phone_addiction_dataset.csv")
df = df_raw.copy()
print("Loaded:", "teen_phone_addiction_dataset.csv")

--------------------------------

### Display and inspect the dataset

In [ ]:
df.head()

In [ ]:
print("Shape:", df.shape)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
df.info()
display(df.describe(include="all").T)

In [ ]:
df.describe(include=["object","str"])

-----------------------------------------------

### Data cleaning and outlier detection
We will look for missing values, duplicate rows, and zeros in measurement columns. 

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
print("Missing values per column:")
display(missing[missing > 0] if missing.any() else pd.Series({"No missing values": 0}))
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate IDs:", df["ID"].duplicated().sum() if "ID" in df else "ID column not present")


noting numerical v/s categorical data.

In [ ]:
numerical_columns = df.select_dtypes(include=np.number).columns
categorical_columns = df.select_dtypes(include=["object","str"]).columns
print("Numerical Columns:")
print(list(numerical_columns))
print("\nCategorical Columns:")
print(list(categorical_columns))

---------------------------------------------

### Checking invalid values.
Values outside a range are flagged for review, not silently deleted.

In [ ]:
range_rules = {
    "Age": (13, 19), "Daily_Usage_Hours": (0, 24), "Sleep_Hours": (0, 24), "Academic_Performance": (0, 100), "Social_Interactions": (0, 10),
    "Exercise_Hours": (0, 24), "Anxiety_Level": (1, 10), "Depression_Level": (1, 10), "Self_Esteem": (1, 10),
    "Parental_Control": (0, 1), "Screen_Time_Before_Bed": (0, 24), "Phone_Checks_Per_Day": (0, None), "Apps_Used_Daily": (0, None),
    "Time_on_Social_Media": (0, 24), "Time_on_Gaming": (0, 24), "Time_on_Education": (0, 24), "Family_Communication": (1, 10), 
    "Weekend_Usage_Hours": (0, 24), "Addiction_Level": (1, 10)
}
invalid = {}
for col, (low, high) in range_rules.items():
    if col in df.columns:
        bad = df[col] < low
        if high is not None:
            bad |= df[col] > high
        if bad.any():
            invalid[col] = df.loc[bad, col].tolist()
print("Out-of-range values:", invalid if invalid else "None found under these screening rules")

----------------------------------------------------------

### histograms and boxplots

In [ ]:
analysis_cols = [
    c for c in df.select_dtypes(include="number").columns
    if c not in ["ID", "Parental_Control"]
]
plot_cols = [c for c in analysis_cols if c in df.columns]
axes = df[plot_cols].hist(figsize=(16, 18), bins=20, edgecolor="white", grid=False)
plt.suptitle("Distributions of numeric features (raw data)", y=1.01, fontsize=16)
plt.tight_layout()
plt.show()

1. Age: Ages 13-19 are represented in fairly similar numbers.
2. Daily usage: Most values cluster around 4-6 hours. very low and very high daily-use values are less common.
3. Sleep: Most values cluster around 6-7.5 hours, with fewer observations at the extremes.
4. Exercise: Values are concentrated at lower hours and get less as exercise time increases.
5. Weekend usage: Most values are around 5-8 hours.
6. Social media, gaming, and education time: Their distributions differ, but the chart alone doesn’t show whether those times are related to addiction scores.
7. Addiction level: Scores pile up at 10, the maximum. This ceiling effect means the score has limited ability to distinguish people at the high end, so interpret relationships involving it cautiously.
8. Rating-style variables such as anxiety, depression, self-esteem, and family communication have roughly similar counts across their values; the chart doesn’t suggest a strong concentration at one score.

-------------------------------------------------------

In [ ]:
rows = int(np.ceil(len(plot_cols) / 3))
fig, axes = plt.subplots(rows, 3, figsize=(16, 4 * rows))
axes = np.array(axes).reshape(-1)
for ax, col in zip(axes, plot_cols):
    sns.boxplot(y=df[col], ax=ax, color="skyblue")
    ax.set_title(col)
for ax in axes[len(plot_cols):]: ax.remove()
fig.suptitle("Boxplots of numeric features (raw data)", y=1.01, fontsize=16)
plt.tight_layout()
plt.show()

1. Daily phone use: Median is about 5 hours. Most values are roughly 3.7-6.4 hours, with a few unusually high observations.
2. Sleep: Median is about 6.5 hours; the middle half is around 5.5-7.5 hours.
3. Exercise: Median is about 1 hour. Most values are below 1.5 hours, with a small number of higher values flagged as outliers.
4. Weekend phone use: Median is about 6 hours, with a few high and low outliers.
5. Addiction_Level: Median is 10, and the middle half is also concentrated near 8-10. Many lower scores appear as outliers because scores cluster at the maximum. This ceiling effect can make comparisons harder.
6. Phone checks, apps, social interactions, and rating scores have fairly broad spreads; there aren’t prominent outliers in most of these plots.

---------------------------------------------------------------

### Pie charts for categorical / binned values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
df["Gender"].value_counts().plot.pie(autopct="%1.1f%%", ax=axes[0], startangle=90)
axes[0].set_title("Gender composition"); axes[0].set_ylabel("")
usage_bins = pd.cut(df["Daily_Usage_Hours"], bins=[-0.01, 3, 6, 9, 24], labels=["0–3", ">3–6", ">6–9", ">9 hours"])
usage_bins.value_counts(sort=False).plot.pie(autopct="%1.1f%%", ax=axes[1], startangle=90)
axes[1].set_title("Daily usage hours (binned)"); axes[1].set_ylabel("")
plt.tight_layout(); plt.show()

- Gender composition: The dataset is nearly evenly split: Male 33.9%, Female 33.6%, and Other 32.6%. Gender representation is balanced overall.
- Daily usage: Most teenagers report more than 3 to 6 hours of daily phone use (53.8%). 28.9% use their phones for more than 6 to 9 hours, 15.7% for 0-3 hours, and 1.7% for over 9 hours.

-----------------------------------------------------------------------

Categorical distributions: bar / count plots

In [ ]:
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
cat_plot_cols = [c for c in cat_cols if c not in ["Name", "Location"]]
fig, axes = plt.subplots(len(cat_plot_cols), 1, figsize=(12, 4 * len(cat_plot_cols)))
if len(cat_plot_cols) == 1: axes = [axes]
for ax, col in zip(axes, cat_plot_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=order, ax=ax, color="steelblue")
    ax.set_title(f"Count by {col}"); ax.set_xlabel("Number of records")
plt.tight_layout(); plt.show()

------------------------------------------------

### Addiction level across categories: boxplots and normalized stacked bars

In [ ]:
group_cols = [c for c in ["Gender", "School_Grade", "Phone_Usage_Purpose", "Parental_Control"] if c in df.columns]
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
for ax, col in zip(axes.flat, group_cols):
    sns.boxplot(data=df, x=col, y="Addiction_Level", ax=ax)
    ax.set_title(f"Addiction_Level by {col}"); ax.tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()

-------------------------

In [ ]:
for col in ["Gender", "School_Grade", "Phone_Usage_Purpose"]:
    if col in df.columns:
        proportions = pd.crosstab(df[col], df["Addiction_Level"] >= 8, normalize="index") * 100
        proportions.columns = ["Below 8", "8 or above"]
        proportions.plot(kind="bar", stacked=True, figsize=(10, 4), color=["#74add1", "#f46d43"])
        plt.title(f"Share with Addiction_Level ≥ 8 by {col}"); plt.ylabel("Percent within category"); plt.xlabel(col)
        plt.xticks(rotation=25); plt.legend(title="Addiction score"); plt.tight_layout(); plt.show()

-----------------

### Relationships between numeric features: correlation heatmap

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
corr_cols = [c for c in numeric_cols if c != "ID"]
corr = df[corr_cols].corr(numeric_only=True)
plt.figure(figsize=(15, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f", square=True, annot_kws={"size": 7})
plt.title("Correlation between numeric features (raw data)")
plt.tight_layout(); plt.show()
print("Correlations with Addiction_Level:")
display(corr["Addiction_Level"].drop("Addiction_Level").sort_values(ascending=False).to_frame("Pearson r"))

1. Daily_Usage_Hours: +0.60 - the clearest relationship in this chart. More daily phone use tends to go with a higher addiction score.
2. Apps_Used_Daily: +0.32, Time_on_Social_Media: +0.31, Time_on_Gaming: +0.27, and Phone_Checks_Per_Day: +0.25 -weaker positive relationships.
3. Sleep_Hours: -0.22 - a weak negative relationship: more sleep tends to go with a slightly lower addiction score.
4. Most other features are close to zero, so this chart shows little linear relationship between them and the score.

------------------------

### Other useful views

In [ ]:
sns.regplot(data=df, x="Daily_Usage_Hours", y="Addiction_Level", scatter_kws={"alpha": 0.25}, line_kws={"color": "red"})
plt.title("Daily phone use and addiction level"); plt.tight_layout(); plt.show()

pair_cols = [c for c in ["Daily_Usage_Hours", "Sleep_Hours", "Phone_Checks_Per_Day", "Addiction_Level"] if c in df.columns]
sns.pairplot(df[pair_cols].sample(min(600, len(df)), random_state=42), corner=True, plot_kws={"alpha": 0.35, "s": 18})
plt.show()

---------------------

### Main findings from this file

In [ ]:
print(f"Rows: {len(df):,}; columns: {df.shape[1]}")
print("Missing cells:", int(df.isna().sum().sum()))
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Mean daily phone use (hours):", round(df["Daily_Usage_Hours"].mean(), 2))
print("Median daily phone use (hours):", round(df["Daily_Usage_Hours"].median(), 2))
print("Mean sleep (hours):", round(df["Sleep_Hours"].mean(), 2))
print("Median Addiction_Level:", round(df["Addiction_Level"].median(), 2))
print("Addiction_Level >= 8 share (%):", round((df["Addiction_Level"] >= 8).mean() * 100, 1))
print("Highest positive correlations with Addiction_Level:")
display(corr["Addiction_Level"].drop("Addiction_Level").sort_values(ascending=False).head(5).round(2).to_frame("Pearson r"))
print("Most negative correlations with Addiction_Level:")
display(corr["Addiction_Level"].drop("Addiction_Level").sort_values().head(5).round(2).to_frame("Pearson r"))


--------------------

### Findings

- The dataset has 3000 rows and 15 columns.
- Missing values or duplicate rows: 0.
- Measurement distributions that stood out:

1. The sample covers ages 13–19. Gender counts are fairly balanced.
2. Average daily phone use is about 5.02 hours (median 5.0); average sleep is about 6.49 hours (median 6.5).
Addiction_Level is concentrated near its maximum: median 10, and about 76% of rows score at least 8. This ceiling effect limits how much lower/higher-score differences can be studied.
3. In the heatmap, Daily_Usage_Hours has the clearest positive correlation with Addiction_Level (r ≈ 0.60); Sleep_Hours is negatively correlated (r ≈ −0.22). Apps used, social-media/gaming time, and phone checks have weaker positive correlations (roughly 0.25–0.32). 
4. The IQR rule flagged some valid-looking high/low observations in daily use, exercise, bedtime screen time, weekend use, and addiction scores.